# ERPy quick start

Create an `Epochs` object, inspect it, run the primary detector, and make one audit-ready summary.

All signals, labels, and coordinates in this notebook are deterministic and
synthetic. They do not represent a participant. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Allow this notebook to run from JupyterLab or from the repository root.
HERE = Path.cwd()
EXAMPLES = HERE if (HERE / "_synthetic.py").exists() else HERE / "notebooks" / "examples"
sys.path.insert(0, str(EXAMPLES.resolve()))
REPOSITORY = EXAMPLES.parents[1]
if (REPOSITORY / "ERPy").exists():
    sys.path.insert(0, str(REPOSITORY.resolve()))

import ERPy as ep
import ERPy.viz as viz
from _synthetic import (
    make_detection_table,
    make_edges,
    make_electrode_metadata,
    make_epochs,
    make_metric_table,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})

In [ ]:
epochs = make_epochs()
print(f"Python: {sys.executable}")
print(f"ERPy {ep.__version__}: {epochs.n_trials()} trials, {len(epochs.channels)} channels")
epochs.get_mean_waveform().head(3)

In [ ]:
detections = epochs.detect_erp_all(
    methods=["crp_energy", "peak_amplitude", "rms_response"],
    min_consensus=1,
)
columns = ["channel", "method", "significant", "primary_significant", "primary_classification"]
detections[columns].drop_duplicates().head(12)

In [ ]:
# The `p_crp` field name is retained for compatibility. On the primary
# `crp_energy` row it stores the fixed-window reproducibility value p_R.
primary_columns = [
    "channel", "p_crp", "p_energy", "p_joint", "q_joint",
    "primary_classification", "primary_significant",
]
detections.loc[detections["method"].eq("crp_energy"), primary_columns].head()

The primary result requires both components: fixed-window whole-trial
amplitude-weighted waveform reproducibility $p_R$ and separately demeaned
matched excess energy $p_E$. The displayed four-class label uses the
unadjusted component thresholds; the final response call uses the
family-adjusted joint value and artifact eligibility.
The standalone `crp_significance` method shown later is a different,
data-selected-duration extraction test retained for descriptive comparison.

In [ ]:
fig = epochs.plot.summary("CONTACT_B1", detections=detections)
fig.suptitle("Synthetic response: waveform, trials, and detector context", y=1.02)
plt.show()

The result is an audit aid, not a clinical interpretation. In real work, next
review event alignment, the artifact report, the post-artifact anchor, and the
clean-trial count before using a response label.